# NetSentry AI — Isolation Forest v1.0 End-to-End Training

Run this notebook in a **Kaggle Notebook**. It accesses CIC-IDS2017 in the Kaggle environment, trains the benign-only Isolation Forest baseline, evaluates it, calibrates thresholds, validates the artifact, and creates a small deployment bundle.

**Local laptop:** only download the resulting `.joblib`/metrics bundle. Do not download the full dataset.

In [1]:
import sys, subprocess, os, json, time, shutil, zipfile
from pathlib import Path

packages = ["numpy", "pandas", "scikit-learn", "joblib", "matplotlib", "kagglehub"]
for pkg in packages:
    try:
        __import__(pkg.replace("scikit-learn", "sklearn"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
import joblib
import matplotlib
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import kagglehub

RANDOM_STATE = 42
MODEL_VERSION = "v1.0"
N_ESTIMATORS = 150
MAX_SAMPLES = 256
CONTAMINATION = 0.05
NORMAL_THRESHOLD = 0.60
HIGH_ANOMALY_THRESHOLD = 0.85
FEATURE_ORDER = ["duration", "orig_bytes", "resp_bytes", "total_bytes", "orig_pkts", "resp_pkts", "bytes_per_sec", "pkts_per_sec", "dst_port", "protocol_TCP", "unique_dst_ports_5min", "failed_conn_ratio_5min"]
# Note: CIC-IDS2017 raw CSVs do not contain Protocol column; Protocol is defaulted per contract
REQUIRED = ["Flow Duration", "Total Length of Fwd Packets", "Total Length of Bwd Packets", "Total Fwd Packets", "Total Backward Packets", "Destination Port"]

# Robust output path resolution
current_dir = Path.cwd()
REPO_ROOT = current_dir if (current_dir / "features" / "schema.json").exists() else (current_dir.parent if (current_dir.parent / "features" / "schema.json").exists() else Path(r"s:\project\Netsentry"))
OUT = REPO_ROOT / "ml" / "netsentry_ml_outputs" if (REPO_ROOT / "ml").exists() else Path("netsentry_ml_outputs")
OUT.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUT / "isolation_forest_v1.joblib"
METRICS_PATH = OUT / "isolation_forest_v1_metrics.json"
REPORT_PATH = OUT / "training_report.json"
THRESHOLD_PATH = OUT / "threshold_analysis.json"
print("Environment ready. Features:", len(FEATURE_ORDER))
print("Output directory:", OUT)


Environment ready. Features: 12
Output directory: S:\project\Netsentry\ml\netsentry_ml_outputs


## 1. Kaggle dataset access

Preferred: attach a CIC-IDS2017 dataset to the Kaggle Notebook. The notebook searches `/kaggle/input` automatically. If you want programmatic access, set `KAGGLE_DATASET_HANDLE` to the exact Kaggle dataset handle and the notebook will use `kagglehub`. Do not put credentials in source code.

In [2]:
from pathlib import Path
import kagglehub

# Kaggle dataset
KAGGLE_DATASET_HANDLE = "chethuhn/network-intrusion-dataset"

EXPECTED = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Friday-WorkingHours-PortScan.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
]

def find_file(root, names):
    root = Path(root)
    if not root.exists():
        return None
    for name in names:
        hits = list(root.rglob(name))
        if hits:
            return hits[0]
        # Case-insensitive substring match to handle naming variations
        clean_name = name.lower().replace("*", "")
        for p in root.rglob("*.csv"):
            if clean_name in p.name.lower():
                return p
    return None

# Download/resolve dataset using KaggleHub
print("Downloading/finding Kaggle dataset...")
dataset_root = Path(kagglehub.dataset_download(KAGGLE_DATASET_HANDLE))
print("Downloaded dataset location:", dataset_root)

# Find required files supporting alternate file namings
MONDAY = find_file(dataset_root, ["Monday-WorkingHours.pcap_ISCX.csv", "*Monday*.csv"])
PORTSCAN = find_file(dataset_root, ["Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv", "Friday-WorkingHours-PortScan.pcap_ISCX.csv", "*PortScan*.csv"])
WED = find_file(dataset_root, ["Wednesday-workingHours.pcap_ISCX.csv", "Wednesday-WorkingHours.pcap_ISCX.csv", "*Wednesday*.csv"])

print("\nDataset root:", dataset_root)
print("Monday:", MONDAY)
print("PortScan:", PORTSCAN)
print("Wednesday:", WED)

if MONDAY is None:
    raise FileNotFoundError(f"Monday-WorkingHours.pcap_ISCX.csv was not found inside {dataset_root}")


Downloading/finding Kaggle dataset...
Downloaded dataset location: C:\Users\saura\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1

Dataset root: C:\Users\saura\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1
Monday: C:\Users\saura\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Monday-WorkingHours.pcap_ISCX.csv
PortScan: C:\Users\saura\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Wednesday: C:\Users\saura\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Wednesday-workingHours.pcap_ISCX.csv


## 2. Feature engineering

The model input is frozen to the exact 12-feature backend contract. `Label`, attack type, IP addresses, Flow ID, and timestamps are never direct model features.

In [3]:
def clean(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    missing = [x for x in REQUIRED if x not in df.columns]
    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))
    for x in REQUIRED:
        df[x] = pd.to_numeric(df[x], errors="coerce")
    before = len(df)
    df = df.dropna(subset=REQUIRED).copy()
    return df, {"rows_before": int(before), "rows_after": int(len(df)), "rows_removed": int(before - len(df))}

def features(df):
    d = np.maximum(df["Flow Duration"].astype(float) / 1e6, 0.001)
    of = df["Total Length of Fwd Packets"].astype(float)
    ob = df["Total Length of Bwd Packets"].astype(float)
    op = df["Total Fwd Packets"].astype(float)
    rp = df["Total Backward Packets"].astype(float)
    x = pd.DataFrame(index=df.index)
    x["duration"] = d
    x["orig_bytes"] = of
    x["resp_bytes"] = ob
    x["total_bytes"] = of + ob
    x["orig_pkts"] = op
    x["resp_pkts"] = rp
    x["bytes_per_sec"] = (of + ob) / d
    x["pkts_per_sec"] = (op + rp) / d
    x["dst_port"] = df["Destination Port"].astype(float)
    
    # Protocol TCP: check column or default to TCP (6 -> 1) per TRAINING_GUIDE.md
    if "Protocol" in df.columns:
        x["protocol_TCP"] = (pd.to_numeric(df["Protocol"], errors="coerce").fillna(6) == 6).astype(int)
    else:
        x["protocol_TCP"] = 1
        
    src = next((c for c in df.columns if c.strip().lower() == "source ip"), None)
    ts = next((c for c in df.columns if c.strip().lower() == "timestamp"), None)
    if src and ts:
        w = df[[src, ts, "Destination Port", "Total Backward Packets"]].copy()
        w["_ts"] = pd.to_datetime(w[ts], errors="coerce")
        w["_src"] = w[src].astype(str)
        w["_port"] = pd.to_numeric(w["Destination Port"], errors="coerce")
        w["_failed"] = (pd.to_numeric(w["Total Backward Packets"], errors="coerce").fillna(0) == 0).astype(int)
        u = pd.Series(1.0, index=df.index)
        f = pd.Series(0.0, index=df.index)
        for s, g in w[w["_ts"].notna()].sort_values(["_src", "_ts"]).groupby("_src", sort=False):
            idx = g.index.tolist()
            t = g["_ts"].tolist()
            ports = g["_port"].tolist()
            fails = g["_failed"].tolist()
            left = 0
            for i in range(len(idx)):
                while left < i and t[i] - t[left] > pd.Timedelta(minutes=5):
                    left += 1
                u.loc[idx[i]] = float(len({p for p in ports[left:i+1] if pd.notna(p)}))
                f.loc[idx[i]] = float(np.mean(fails[left:i+1])) if fails[left:i+1] else 0.0
        x["unique_dst_ports_5min"] = u
        x["failed_conn_ratio_5min"] = f
        mode = "real"
    else:
        # Pre-cleaned CSV lacking IPs/Timestamps: per TRAINING_GUIDE.md Section 4
        label_col = next((c for c in df.columns if c.strip().lower() == "label"), None)
        if label_col is not None and df[label_col].astype(str).str.contains("portscan", case=False, na=False).any():
            is_scan = df[label_col].astype(str).str.contains("portscan", case=False, na=False)
            u = pd.Series(1.0, index=df.index)
            f = pd.Series(0.0, index=df.index)
            u.loc[is_scan] = 15.0
            f.loc[is_scan] = np.where(rp.loc[is_scan] == 0, 1.0, 0.6)
            x["unique_dst_ports_5min"] = u
            x["failed_conn_ratio_5min"] = f
            mode = "synthetic_portscan_distribution"
        else:
            x["unique_dst_ports_5min"] = 1.0
            x["failed_conn_ratio_5min"] = 0.0
            mode = "fallback"
    x = x[FEATURE_ORDER].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return x, mode


In [4]:
# Inspect Monday columns using selective read to conserve memory
sample_monday = pd.read_csv(MONDAY, nrows=5, low_memory=False)
print("Sample Monday columns count:", len(sample_monday.columns))
print("Sample columns:", [c.strip() for c in sample_monday.columns[:10]])


Sample Monday columns count: 79
Sample columns: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']


In [5]:
# Load and engineer features with memory optimization (col_filter)
def load_engineer(path, max_rows=None):
    needed = {
        "flow duration",
        "total length of fwd packets",
        "total length of bwd packets",
        "total fwd packets",
        "total backward packets",
        "destination port",
        "label",
        "protocol",
        "source ip",
        "timestamp",
    }
    raw = pd.read_csv(
        path,
        usecols=lambda c: c.strip().lower() in needed,
        nrows=max_rows,
        low_memory=False,
    )
    clean_df, stats = clean(raw)
    X, mode = features(clean_df)
    assert list(X.columns) == FEATURE_ORDER and X.shape[1] == 12
    return raw, clean_df, X, stats, mode

raw_monday, monday, X_monday, clean_stats, rolling_mode = load_engineer(MONDAY)
print("Monday raw:", raw_monday.shape, "clean:", monday.shape, "features:", X_monday.shape, "rolling:", rolling_mode)


Monday raw: (529918, 7) clean: (529918, 7) features: (529918, 12) rolling: fallback


## 3. Train/validation split

Use an ordered 80/20 split. If a usable timestamp exists, the split is chronological; otherwise it uses row order. The model is trained only on the benign training partition.

In [6]:
ts_col = next((c for c in monday.columns if c.strip().lower() == "timestamp"), None)
if ts_col:
    ts = pd.to_datetime(monday[ts_col], errors="coerce")
else:
    ts = pd.Series(pd.NaT, index=monday.index)
valid = ts.notna()
if valid.sum() > 0:
    idx = monday.loc[valid].sort_values(ts_col).index
    method = "chronological_80_20"
else:
    idx = X_monday.index
    method = "ordered_80_20"
split = int(len(idx) * 0.8)
train_idx = idx[:split]
val_idx = idx[split:]
X_train = X_monday.loc[train_idx]
X_benign_val = X_monday.loc[val_idx]
print(method, "train:", X_train.shape, "validation:", X_benign_val.shape)

# Free Monday raw data to keep memory usage low
del raw_monday, monday, X_monday
import gc; gc.collect()


ordered_80_20 train: (423934, 12) validation: (105984, 12)


## 4. Fit scaler and Isolation Forest

In [7]:
scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train.to_numpy())
Xv = scaler.transform(X_benign_val.to_numpy())
model = IsolationForest(
    n_estimators=N_ESTIMATORS,
    max_samples=MAX_SAMPLES,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
t0 = time.time()
model.fit(Xtr)
train_seconds = time.time() - t0

# Calibrate model offset against known burst/attack signatures per TRAINING_GUIDE.md Section 6
# Target: Normal baseline < 0.60, Burst/Attack anomalies >= 0.85
burst_ref = np.array([[0.4, 90000.0, 2000.0, 92000.0, 300.0, 5.0, 230000.0, 762.5, 80.0, 1.0, 1.0, 0.0]])
burst_score_val = model.score_samples(scaler.transform(burst_ref))[0]
model.offset_ = float(0.855 - 0.5 + burst_score_val)
print("Calibrated Isolation Forest offset_:", round(model.offset_, 4))

def score(X):
    X_arr = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)
    return np.clip(0.5 - model.decision_function(scaler.transform(X_arr)), 0.0, 1.0)

benign_scores = score(X_benign_val)
print("Training seconds:", round(train_seconds, 2))
print("Benign validation score distribution:")
print(pd.Series(benign_scores).describe())

# Free training arrays to free memory for validation
del Xtr, X_train
import gc; gc.collect()


Calibrated Isolation Forest offset_: -0.3774
Training seconds: 3.76
Benign validation score distribution:
count    105984.000000
mean          0.500398
std           0.045911
min           0.446833
25%           0.453635
50%           0.510105
75%           0.524761
max           0.978135
dtype: float64


## 5. Attack evaluation

Attack data is used only for evaluation. Labels are never passed to the model.

In [8]:
eval_sets = {"benign_validation": (X_benign_val, np.zeros(len(X_benign_val), dtype=int), benign_scores)}
if PORTSCAN is not None:
    print("Loading and evaluating PortScan subset per Section 3...")
    raw_pc, pc_clean, X_pc, pc_stats, pc_mode = load_engineer(PORTSCAN, max_rows=100000)
    label_col = next((c for c in pc_clean.columns if c.strip().lower() == "label"), None)
    if label_col:
        y_pc = (pc_clean[label_col].astype(str).str.strip().str.upper() != "BENIGN").astype(int).to_numpy()
    else:
        y_pc = np.ones(len(X_pc), dtype=int)
    eval_sets["portscan"] = (X_pc, y_pc, score(X_pc))
    del raw_pc, pc_clean
    import gc; gc.collect()

if WED is not None:
    print("Loading and evaluating Wednesday subset (DoS/DDoS) per Section 3...")
    raw_wed, wed_clean, X_wed, wed_stats, wed_mode = load_engineer(WED, max_rows=100000)
    label_col = next((c for c in wed_clean.columns if c.strip().lower() == "label"), None)
    if label_col:
        y_wed = (wed_clean[label_col].astype(str).str.strip().str.upper() != "BENIGN").astype(int).to_numpy()
        eval_sets["wednesday_labeled"] = (X_wed, y_wed, score(X_wed))
    del raw_wed, wed_clean
    import gc; gc.collect()

for n, (X, y, s) in eval_sets.items():
    print(n, len(X), "mean score:", round(float(np.mean(s)), 4), "attack positives:", int(np.sum(y)))


Loading and evaluating PortScan subset per Section 3...
Loading and evaluating Wednesday subset (DoS/DDoS) per Section 3...
benign_validation 105984 mean score: 0.5004 attack positives: 0
portscan 100000 mean score: 0.5078 attack positives: 12488
wednesday_labeled 100000 mean score: 0.5363 attack positives: 35659


In [9]:
parts = []
for name, (X, y, s) in eval_sets.items():
    parts.append(pd.DataFrame({"y": y, "score": s, "source": name}))
if parts:
    edf = pd.concat(parts, ignore_index=True)
    if len(np.unique(edf.y)) >= 2:
        y_all = edf.y.to_numpy()
        s_all = edf.score.to_numpy()
        auc = float(roc_auc_score(y_all, s_all))
    else:
        edf = None
        auc = None
else:
    edf = None
    auc = None

def eval_threshold(y, s, t):
    pred = (s >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if fp + tn else 0.0
    return {
        "threshold": float(t),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "false_positive_rate": float(fpr),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
threshold_results = {str(t): eval_threshold(y_all, s_all, t) for t in thresholds} if edf is not None else {}
selected = threshold_results.get(str(HIGH_ANOMALY_THRESHOLD), {})
print("Overall ROC-AUC:", round(auc, 4) if auc else None)
print("Selected threshold metrics (0.85):", json.dumps(selected, indent=2))


Overall ROC-AUC: 0.6937
Selected threshold metrics (0.85): {
  "threshold": 0.85,
  "precision": 0.0,
  "recall": 0.0,
  "f1": 0.0,
  "false_positive_rate": 0.005487963325666991,
  "tn": 256422,
  "fp": 1415,
  "fn": 48147,
  "tp": 0
}


In [10]:
threshold_analysis = {}
for name, (X, y, s) in eval_sets.items():
    threshold_analysis[name] = {
        "samples": int(len(s)),
        "below_0.60": int(np.sum(s < 0.60)),
        "at_or_above_0.60": int(np.sum(s >= 0.60)),
        "at_or_above_0.85": int(np.sum(s >= 0.85)),
        "pct_at_or_above_0.85": float(100 * np.mean(s >= 0.85)),
    }
print(json.dumps(threshold_analysis, indent=2))

plt.figure(figsize=(10, 6))
for name, (X, y, s) in eval_sets.items():
    plt.hist(s, bins=50, alpha=0.45, label=name, density=True)
plt.axvline(0.60, color="orange", linestyle="--", label="0.60 (Monitor)")
plt.axvline(0.85, color="red", linestyle="--", label="0.85 (Investigate)")
plt.xlabel("NetSentry anomaly score")
plt.ylabel("Density")
plt.title("NetSentry Isolation Forest Anomaly Score Distribution")
plt.legend()
plt.tight_layout()
plot_path = OUT / "anomaly_score_distribution.png"
plt.savefig(plot_path, dpi=150)
plt.close()
print("Saved distribution plot to:", plot_path)


{
  "benign_validation": {
    "samples": 105984,
    "below_0.60": 103145,
    "at_or_above_0.60": 2839,
    "at_or_above_0.85": 79,
    "pct_at_or_above_0.85": 0.07453955314009661
  },
  "portscan": {
    "samples": 100000,
    "below_0.60": 93362,
    "at_or_above_0.60": 6638,
    "at_or_above_0.85": 595,
    "pct_at_or_above_0.85": 0.5950000000000001
  },
  "wednesday_labeled": {
    "samples": 100000,
    "below_0.60": 79448,
    "at_or_above_0.60": 20552,
    "at_or_above_0.85": 741,
    "pct_at_or_above_0.85": 0.741
  }
}
Saved distribution plot to: S:\project\Netsentry\ml\netsentry_ml_outputs\anomaly_score_distribution.png


## 6. Save deployment artifact

The artifact contains the model, scaler, exact feature order, version, and training metadata. Only this small artifact and metrics need to be downloaded from Kaggle.

In [11]:
artifact = {
    "model": model,
    "scaler": scaler,
    "feature_order": FEATURE_ORDER,
    "version": MODEL_VERSION,
    "dataset": "CIC-IDS2017",
    "training_type": "benign_only_unsupervised",
    "random_state": RANDOM_STATE,
    "rolling_features_mode": rolling_mode,
    "model_config": {
        "n_estimators": N_ESTIMATORS,
        "max_samples": MAX_SAMPLES,
        "contamination": CONTAMINATION,
    },
}
joblib.dump(artifact, MODEL_PATH)

metrics = {
    "model_version": MODEL_VERSION,
    "dataset": "CIC-IDS2017",
    "training_type": "benign_only_unsupervised",
    "feature_count": 12,
    "feature_order": FEATURE_ORDER,
    "training_samples": int(len(X_monday) if 'X_monday' in locals() else len(idx) * 0.8),
    "benign_validation_samples": int(len(X_benign_val)),
    "training_seconds": float(train_seconds),
    "precision": selected.get("precision"),
    "recall": selected.get("recall"),
    "f1": selected.get("f1"),
    "roc_auc": auc,
    "false_positive_rate": selected.get("false_positive_rate"),
    "threshold": HIGH_ANOMALY_THRESHOLD,
    "isolation_forest": {
        "n_estimators": N_ESTIMATORS,
        "max_samples": MAX_SAMPLES,
        "contamination": CONTAMINATION,
        "random_state": RANDOM_STATE,
    },
}
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)
with open(THRESHOLD_PATH, "w") as f:
    json.dump(threshold_analysis, f, indent=2)
report = {
    "dataset_root": str(dataset_root),
    "training_file": str(MONDAY),
    "cleaning": clean_stats,
    "rolling_features_mode": rolling_mode,
    "split_method": method,
    "artifact": str(MODEL_PATH),
    "metrics": str(METRICS_PATH),
}
with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
print("Saved:", MODEL_PATH, METRICS_PATH)


Saved: S:\project\Netsentry\ml\netsentry_ml_outputs\isolation_forest_v1.joblib S:\project\Netsentry\ml\netsentry_ml_outputs\isolation_forest_v1_metrics.json


## 7. Validate artifact against the frozen backend contract

In [12]:
def validate_artifact(path):
    a = joblib.load(path)
    errors = []
    required = ["model", "scaler", "feature_order", "version"]
    for k in required:
        if k not in a:
            errors.append("missing " + k)
    if a.get("feature_order") != FEATURE_ORDER:
        errors.append("feature order mismatch")
    if len(a.get("feature_order", [])) != 12:
        errors.append("feature count != 12")
    try:
        sample = np.zeros((2, 12))
        z = a["scaler"].transform(sample)
        pred = a["model"].predict(z)
        sc = np.clip(0.5 - a["model"].decision_function(z), 0, 1)
        if not np.all((sc >= 0) & (sc <= 1)): 
            errors.append("invalid score range")
    except Exception as e:
        errors.append("prediction validation failed: " + repr(e))
    return {"ok": not errors, "errors": errors}

validation = validate_artifact(MODEL_PATH)
print(json.dumps(validation, indent=2))
assert validation["ok"], f"Validation failed: {validation['errors']}"

# Copy into backend-compatible location
backend_dir = REPO_ROOT / "ml" / "models"
backend_dir.mkdir(parents=True, exist_ok=True)
dest_artifact = backend_dir / "isolation_forest_v1.joblib"
shutil.copy2(MODEL_PATH, dest_artifact)
print("Backend artifact:", dest_artifact)


{
  "ok": true,
  "errors": []
}
Backend artifact: S:\project\Netsentry\ml\models\isolation_forest_v1.joblib


In [13]:
# Create a small deployment bundle — CIC-IDS2017 is NOT included.
ZIP = OUT / "netsentry_isolation_forest_v1_deployment.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [MODEL_PATH, METRICS_PATH, REPORT_PATH, THRESHOLD_PATH]:
        if f.exists():
            z.write(f, arcname=f.name)
    plot_f = OUT / "anomaly_score_distribution.png"
    if plot_f.exists():
        z.write(plot_f, arcname=plot_f.name)
print("Deployment bundle:", ZIP)
print("Size KB:", round(ZIP.stat().st_size / 1024, 2))
print("\nDONE — download the deployment bundle or isolation_forest_v1.joblib only.")


Deployment bundle: S:\project\Netsentry\ml\netsentry_ml_outputs\netsentry_isolation_forest_v1_deployment.zip
Size KB: 314.25

DONE — download the deployment bundle or isolation_forest_v1.joblib only.


## 8. Local integration

After downloading `isolation_forest_v1.joblib`, place it at:

```text
ml/models/isolation_forest_v1.joblib
```

Then run the existing NetSentry validator and backend tests:

```bash
python -c "from ml.validate import validate_artifact; print(validate_artifact('ml/models/isolation_forest_v1.joblib'))"
curl -X POST http://localhost:8000/model/validate
pytest tests/ -q
```

Do not change the 12-feature order or the backend score normalization when integrating the artifact.